# The Bedrock Converse API — Masterclass

One API, every provider. A reference for: single-turn, multi-turn, streaming, manual control, tool use with every useful variation, multimodal, production exception handling, and CloudWatch logging + observability.

**Why Converse over InvokeModel:** `InvokeModel` makes you hand-build a different JSON body and parse a different response for every provider. `converse` gives one structure across all of them — swapping models is a one-line `modelId` change.

> IAM note: Converse is authorized under **`bedrock:InvokeModel`**. `bedrock:Converse` is not a valid IAM action.

## Anatomy

**Request**

| Field | What it is |
|---|---|
| `modelId` | The model or `us.` inference-profile ID |
| `messages` | The conversation: list of `{role, content}`, role is `user` or `assistant` |
| `system` | Optional list of system blocks: `[{"text": "..."}]` |
| `inferenceConfig` | `maxTokens`, `temperature`, `topP`, `stopSequences` |
| `toolConfig` | Tools the model may call + `toolChoice` |

**Content blocks** inside `content`: `text`, `image`, `document`, `toolUse`, `toolResult`.

**Response**

| Field | What it is |
|---|---|
| `output.message.content` | The model's reply blocks |
| `stopReason` | `end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `guardrail_intervened`, `content_filtered` |
| `usage` | `inputTokens`, `outputTokens`, `totalTokens` |
| `metrics.latencyMs` | Server-side latency |
| `ResponseMetadata.RequestId` | Trace id for support / logs |

## Setup (VS Code and Colab)

**VS Code:** venv → install cell → `aws configure` or env vars (`AWS_DEFAULT_REGION=us-east-1`) → select venv kernel.
**Colab:** install cell → set creds via `os.environ`/secrets → run.

In [1]:
%pip install -q boto3 pillow

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json, time, random, sys, logging
import datetime as dt
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError, NoCredentialsError

REGION = "us-east-1"
NOVA_LITE = "us.amazon.nova-2-lite-v1:0"
HAIKU_45  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET_45 = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

MODEL_ID = HAIKU_45            # default anchor (Claude supports every toolChoice mode)
PRICE = {NOVA_LITE: (0.30, 2.50), HAIKU_45: (1.00, 5.00), SONNET_45: (3.00, 15.00)}

bedrock = boto3.client(
    "bedrock-runtime", region_name=REGION,
    config=Config(retries={"max_attempts": 5, "mode": "adaptive"},
                  read_timeout=120, connect_timeout=10),
)

def preflight():
    try:
        boto3.client("sts", region_name=REGION).get_caller_identity()
    except NoCredentialsError:
        print("No AWS credentials. Run `aws configure` or set env vars.")
        return
    try:
        bedrock.converse(modelId=MODEL_ID,
                         messages=[{"role": "user", "content": [{"text": "ping"}]}],
                         inferenceConfig={"maxTokens": 5})
        print(f"OK: {MODEL_ID} reachable in {REGION}.")
    except ClientError as e:
        print("Converse failed:", e.response["Error"]["Code"])

preflight()

OK: us.anthropic.claude-haiku-4-5-20251001-v1:0 reachable in us-east-1.


## 1. Single-turn

The minimal call. Read `output.message.content`, plus `stopReason`, `usage`, and `metrics`.

In [5]:
resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": "Name three Star Alliance airlines."}]}],
    inferenceConfig={"maxTokens": 100})

print(resp["output"]["message"]["content"][0]["text"])
print("stopReason:", resp["stopReason"],
      "| usage:", resp["usage"],
      "| latencyMs:", resp["metrics"]["latencyMs"])

Three Star Alliance airlines are:

1. **Lufthansa** (Germany)
2. **United Airlines** (United States)
3. **Singapore Airlines** (Singapore)
stopReason: end_turn | usage: {'inputTokens': 13, 'outputTokens': 42, 'totalTokens': 55, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0} | latencyMs: 1006


### System prompt and the inference knobs

- `temperature` (0-1): 0 = deterministic, higher = more varied. Use 0 for extraction/rules.
- `topP`: nucleus sampling; tune one of temperature/topP, not both.
- `maxTokens`: hard cap on output — your biggest cost lever.
- `stopSequences`: strings that end generation early.

In [ ]:
# Demo: ask for a list but stop generation right before item 4.
resp = bedrock.converse(
    modelId=MODEL_ID,
    system=[{"text": "You are a terse airline ops assistant."}],
    messages=[{"role": "user", "content": [{"text": "List airline alliance benefits, numbered."}]}],
    inferenceConfig={"maxTokens": 200, "temperature": 0.0,
                     "stopSequences": ["4."]})
print(resp["output"]["message"]["content"][0]["text"])
print("stopReason:", resp["stopReason"])   # -> stop_sequence

ValidationException: An error occurred (ValidationException) when calling the Converse operation: The model returned the following errors: `temperature` and `top_p` cannot both be specified for this model. Please use only one.

## 2. Multi-turn — you manage the history

The API is **stateless**: it remembers nothing between calls. To hold a conversation, resend the full `messages` list every time, appending each new turn. Append the model's reply (`role: assistant`) and the user's next message (`role: user`).

In [ ]:
messages = [{"role": "user", "content": [{"text": "I'm flying BLR to SIN next week."}]}]
resp = bedrock.converse(modelId=MODEL_ID, messages=messages,
                        inferenceConfig={"maxTokens": 120})
messages.append(resp["output"]["message"])                       # keep assistant turn
print("A:", resp["output"]["message"]["content"][0]["text"])

messages.append({"role": "user",
                 "content": [{"text": "Roughly how long is that flight?"}]})
resp = bedrock.converse(modelId=MODEL_ID, messages=messages,
                        inferenceConfig={"maxTokens": 120})
print("A:", resp["output"]["message"]["content"][0]["text"])  # remembers BLR->SIN, we resent it

### A reusable stateful wrapper

The same pattern as a small class — append on the way in and out.

In [ ]:
class Chat:
    """Minimal stateful wrapper over the stateless API."""
    def __init__(self, model_id=MODEL_ID, system=None):
        self.model_id = model_id
        self.system = [{"text": system}] if system else None
        self.messages = []
    def ask(self, text, max_tokens=300, temperature=0.3):
        self.messages.append({"role": "user", "content": [{"text": text}]})
        kw = dict(modelId=self.model_id, messages=self.messages,
                  inferenceConfig={"maxTokens": max_tokens, "temperature": temperature})
        if self.system:
            kw["system"] = self.system
        resp = bedrock.converse(**kw)
        self.messages.append(resp["output"]["message"])
        return "".join(b.get("text", "") for b in resp["output"]["message"]["content"])

c = Chat(system="You are a helpful airline assistant.")
print(c.ask("My PNR is JX48Q2."))
print(c.ask("Remind me what my PNR was."))   # carries context

## 3. Streaming — `converse_stream`

For chat UIs you want tokens as they are generated, not after. `converse_stream` returns an event stream. The useful events: `contentBlockDelta` (text chunks), `messageStop` (the stop reason), and `metadata` (final `usage` and `metrics`).

In [ ]:
stream = bedrock.converse_stream(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [
        {"text": "Write a 3-sentence apology for a delayed flight."}]}],
    inferenceConfig={"maxTokens": 200})

usage = None
for event in stream["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"].get("text", ""), end="", flush=True)
    elif "messageStop" in event:
        print()
        print(f"[stopReason: {event['messageStop']['stopReason']}]")
    elif "metadata" in event:
        usage = event["metadata"]["usage"]
print("usage:", usage)

## 4. Tool use — the heart of agents

A tool is a JSON-schema contract you give the model. The model never runs anything; it *requests* a call (`stopReason == "tool_use"`, a `toolUse` block), you run the function, and you return a `toolResult`.

`toolChoice` controls how eager it is:
- `{"auto": {}}` — model decides (default)
- `{"any": {}}` — must call some tool
- `{"tool": {"name": "X"}}` — must call tool X (great for forcing structured output)

> Claude models support all three. Some models support only `auto`/`any` — keep the default for portability and force tools only when needed.

### One useful tool, many shapes

We build a flight tool for TravelMind and walk every variation you will actually use: required params, optional defaults, enums, nested objects, arrays, error results the model recovers from, forced structured output, and parallel calls.

First, a tiny generic runner so each variation stays short.

In [ ]:
def run_once(user_text, tools, fns, system=None, force=None, max_tokens=800):
    tc = {"tools": tools, "toolChoice": force or {"auto": {}}}
    msgs = [{"role": "user", "content": [{"text": user_text}]}]
    kw = dict(modelId=MODEL_ID, messages=msgs, toolConfig=tc,
              inferenceConfig={"maxTokens": max_tokens, "temperature": 0.0})
    if system:
        kw["system"] = [{"text": system}]
    resp = bedrock.converse(**kw)
    msgs.append(resp["output"]["message"])
    if resp["stopReason"] != "tool_use":
        return "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    results = []
    for b in resp["output"]["message"]["content"]:
        if "toolUse" in b:
            tu = b["toolUse"]
            print(f"  model called {tu['name']}({json.dumps(tu['input'])})")
            out = fns[tu["name"]](**tu["input"])
            results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                            "content": [{"json": out}], "status": "success"}})
    msgs.append({"role": "user", "content": results})
    resp = bedrock.converse(modelId=MODEL_ID, messages=msgs, toolConfig=tc,
                            inferenceConfig={"maxTokens": max_tokens, "temperature": 0.0})
    return "".join(b.get("text", "") for b in resp["output"]["message"]["content"])

**Variation 1 — required params (the simplest tool).**

In [ ]:
def search_flights(origin, destination, date):
    """DUMMY flight search."""
    return {"results": [
        {"flight": "TM482", "dep": "09:15", "arr": "15:40", "price_inr": 18500},
        {"flight": "TM618", "dep": "20:05", "arr": "02:30+1", "price_inr": 14200}]}

SEARCH_TOOL = {"toolSpec": {
    "name": "search_flights",
    "description": "Search available flights between two airports on a date.",
    "inputSchema": {"json": {
        "type": "object",
        "properties": {
            "origin": {"type": "string", "description": "IATA code, e.g. BLR"},
            "destination": {"type": "string", "description": "IATA code, e.g. SIN"},
            "date": {"type": "string", "description": "YYYY-MM-DD"}},
        "required": ["origin", "destination", "date"]}}}}

print(run_once("Find flights from Bangalore to Singapore on 2026-08-14.",
               [SEARCH_TOOL], {"search_flights": search_flights}))

**Variation 2 — optional params + enums.** Make a param optional by leaving it out of `required`; give it a default in the function. Constrain values with `enum` so the model cannot pass garbage.

In [ ]:
def search_flights_v2(origin, destination, date, cabin="economy", max_price_inr=None):
    pool = [{"flight": "TM482", "cabin": "economy", "price_inr": 18500},
            {"flight": "TM900", "cabin": "business", "price_inr": 52000}]
    out = [f for f in pool if f["cabin"] == cabin]
    if max_price_inr:
        out = [f for f in out if f["price_inr"] <= max_price_inr]
    return {"results": out}

SEARCH_TOOL_V2 = {"toolSpec": {
    "name": "search_flights_v2",
    "description": "Search flights with optional cabin and price filters.",
    "inputSchema": {"json": {
        "type": "object",
        "properties": {
            "origin": {"type": "string"}, "destination": {"type": "string"},
            "date": {"type": "string"},
            "cabin": {"type": "string", "enum": ["economy", "business", "first"],
                      "description": "Defaults to economy if omitted."},
            "max_price_inr": {"type": "integer"}},
        "required": ["origin", "destination", "date"]}}}}

print(run_once("BLR to SIN on 2026-08-14 in business class.",
               [SEARCH_TOOL_V2], {"search_flights_v2": search_flights_v2}))

**Variation 3 — arrays and nested objects.** Real itineraries are multi-segment. Accept a list of segment objects.

In [ ]:
def price_itinerary(segments):
    total = sum(s.get("pax", 1) * 12000 for s in segments)
    return {"segments": segments, "total_inr": total, "stops": len(segments) - 1}

ITIN_TOOL = {"toolSpec": {
    "name": "price_itinerary",
    "description": "Price a multi-segment itinerary.",
    "inputSchema": {"json": {
        "type": "object",
        "properties": {
            "segments": {
                "type": "array",
                "items": {"type": "object",
                    "properties": {"from": {"type": "string"}, "to": {"type": "string"},
                                   "date": {"type": "string"}, "pax": {"type": "integer"}},
                    "required": ["from", "to", "date"]}}},
        "required": ["segments"]}}}}

print(run_once("Price a trip: BLR to DEL on 2026-08-14, DEL to SIN on 2026-08-16, 2 passengers.",
               [ITIN_TOOL], {"price_itinerary": price_itinerary}))

**Variation 4 — tool errors the model recovers from.** Return `status: "error"` with a helpful message; the model reads it and adjusts (re-asks, tries different args, or explains to the user). This runner surfaces the error status back to the model.

In [ ]:
def get_pnr(pnr):
    if pnr != "JX48Q2":
        return {"error": "PNR not found. Confirm the 6-character code."}
    return {"pnr": pnr, "pax": "A. Das", "route": "BLR-SIN", "status": "confirmed"}

PNR_TOOL = {"toolSpec": {
    "name": "get_pnr", "description": "Look up a booking by PNR.",
    "inputSchema": {"json": {"type": "object",
        "properties": {"pnr": {"type": "string"}}, "required": ["pnr"]}}}}

def run_with_errors(user_text, tools, fns, max_turns=4):
    tc = {"tools": tools, "toolChoice": {"auto": {}}}
    msgs = [{"role": "user", "content": [{"text": user_text}]}]
    for _ in range(max_turns):
        resp = bedrock.converse(modelId=MODEL_ID, messages=msgs, toolConfig=tc,
                                inferenceConfig={"maxTokens": 500, "temperature": 0.0})
        msgs.append(resp["output"]["message"])
        if resp["stopReason"] != "tool_use":
            return "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
        results = []
        for b in resp["output"]["message"]["content"]:
            if "toolUse" in b:
                tu = b["toolUse"]
                out = fns[tu["name"]](**tu["input"])
                status = "error" if "error" in out else "success"
                print(f"  {tu['name']}({tu['input']}) -> {status}")
                results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                                "content": [{"json": out}], "status": status}})
        msgs.append({"role": "user", "content": results})
    return "stopped"

print(run_with_errors("Look up PNR ZZZZZZ for me.", [PNR_TOOL], {"get_pnr": get_pnr}))

**Variation 5 — forced structured output (the most useful non-obvious trick).** Define a tool whose only job is to *receive* clean structured data, then force it with `toolChoice`. The model's tool-call arguments become your guaranteed-schema JSON — no fragile parsing of prose.

In [ ]:
EXTRACT_TOOL = {"toolSpec": {
    "name": "record_complaint",
    "description": "Record a structured customer complaint.",
    "inputSchema": {"json": {
        "type": "object",
        "properties": {
            "category": {"type": "string",
                         "enum": ["delay", "baggage", "refund", "seating", "staff", "other"]},
            "severity": {"type": "string", "enum": ["low", "medium", "high"]},
            "pnr": {"type": "string"},
            "summary": {"type": "string"}},
        "required": ["category", "severity", "summary"]}}}}

complaint = ("Absolutely furious. Flight TM482 under PNR JX48Q2 was delayed 6 hours and my "
             "suitcase came out soaked. Nobody at the desk would help.")

resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": complaint}]}],
    toolConfig={"tools": [EXTRACT_TOOL],
                "toolChoice": {"tool": {"name": "record_complaint"}}},  # FORCE it
    inferenceConfig={"maxTokens": 300, "temperature": 0.0})

for b in resp["output"]["message"]["content"]:
    if "toolUse" in b:
        print(json.dumps(b["toolUse"]["input"], indent=2))   # clean structured JSON

**Variation 6 — parallel tool calls.** In one turn the model may request several tools at once. Every loop in this notebook already handles this — it iterates *all* `toolUse` blocks and returns one `toolResult` per call. Always loop; never assume exactly one tool call.

## 5. Multimodal — images and documents

Add an `image` or `document` block alongside your text. Same `converse` call.
- Image formats: png, jpeg, gif, webp.
- Document formats: pdf, csv, doc(x), xls(x), html, txt, md. The `name` must be plain (letters, numbers, spaces).

In [ ]:
from PIL import Image, ImageDraw
img = Image.new("RGB", (520, 160), "white")
d = ImageDraw.Draw(img)
for i, ln in enumerate(["PNR: JX48Q2  Flight: TM482", "BLR -> SIN  14 Aug 2026  Seat 14C"]):
    d.text((16, 24 + i * 48), ln, fill="black")
img.save("mini_ticket.png")
with open("mini_ticket.png", "rb") as f:
    b = f.read()

resp = bedrock.converse(modelId=MODEL_ID,
    messages=[{"role": "user", "content": [
        {"text": "Extract the fields as JSON, then one friendly sentence."},
        {"image": {"format": "png", "source": {"bytes": b}}}]}],
    inferenceConfig={"maxTokens": 300, "temperature": 0.0})
print(resp["output"]["message"]["content"][0]["text"])

# PDF: {"document": {"format": "pdf", "name": "ticket", "source": {"bytes": pdf_bytes}}}

## 6. Production-grade exception handling

Bedrock failures fall into two buckets:
- **Transient** (retry with backoff): `ThrottlingException`, `ServiceUnavailableException`, `ModelTimeoutException`, `InternalServerException`, `ModelNotReadyException`.
- **Permanent** (fix, do not retry): `ValidationException`, `AccessDeniedException`, `ResourceNotFoundException`.

`botocore` adaptive retry mode already retries throttling for you; the wrapper below makes the policy explicit and classifies everything, so you log permanent errors instead of silently looping.

In [ ]:
TRANSIENT = {"ThrottlingException", "ServiceUnavailableException",
             "ModelTimeoutException", "InternalServerException", "ModelNotReadyException"}

def safe_converse(max_retries=4, base_delay=0.5, **kwargs):
    for attempt in range(max_retries + 1):
        try:
            return bedrock.converse(**kwargs)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in TRANSIENT and attempt < max_retries:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 0.3)  # backoff + jitter
                print(f"  transient {code}; retry {attempt + 1}/{max_retries} in {delay:.1f}s")
                time.sleep(delay)
                continue
            raise RuntimeError(f"Converse failed [{code}]: "
                               f"{e.response['Error']['Message']}") from e

resp = safe_converse(modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": "One-line definition of an e-ticket."}]}],
    inferenceConfig={"maxTokens": 60})
print(resp["output"]["message"]["content"][0]["text"])

# PRODUCTION: client-level Config(retries={"mode":"adaptive","max_attempts":5},
#   read_timeout=..., connect_timeout=...) is already set at the top.

## 7. Logging — structured, one line per call

Log what you will actually query later: model, token counts, latency, stop reason, cost, and the `RequestId`. Emit **JSON** so machines can aggregate it.

In [ ]:
logger = logging.getLogger("bedrock")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(h)

def observed_converse(model_id=MODEL_ID, **kwargs):
    resp = safe_converse(modelId=model_id, **kwargs)
    u = resp["usage"]
    p_in, p_out = PRICE.get(model_id, (0, 0))
    record = {
        "ts": dt.datetime.utcnow().isoformat() + "Z",
        "model": model_id,
        "input_tokens": u["inputTokens"], "output_tokens": u["outputTokens"],
        "stop_reason": resp["stopReason"],
        "latency_ms": resp["metrics"]["latencyMs"],
        "cost_usd": round(u["inputTokens"] / 1e6 * p_in + u["outputTokens"] / 1e6 * p_out, 6),
        "request_id": resp["ResponseMetadata"]["RequestId"],
    }
    logger.info(json.dumps(record))
    return resp, record

resp, rec = observed_converse(
    messages=[{"role": "user", "content": [{"text": "Say hi in 5 words."}]}],
    inferenceConfig={"maxTokens": 20})
print()
print("logged record ->", rec)

## 8. CloudWatch — consolidated logging and metrics, the easy way

Three layers, easiest first.

**A. Capture every model call with zero app code — Bedrock model invocation logging.** Flip it on once and Bedrock writes every invocation (prompt, response, token counts) to a CloudWatch log group automatically. The single easiest way to get consolidated usage logs.

**B. Push custom metrics from your app — `put_metric_data`.** For dashboards and alarms on tokens/cost/latency. Works from this notebook right now.

**C. One write for logs + metrics — EMF (Embedded Metric Format).** In production (Lambda/ECS) stdout auto-ships to CloudWatch Logs. Emit EMF-formatted JSON and CloudWatch auto-extracts metrics from the same log line.

The cells below are guarded — they print guidance instead of crashing if permissions are missing.

In [ ]:
# A. One-time setup. Needs a role that lets Bedrock write to the log group.
def enable_bedrock_logging(log_group="/bedrock/travelmind", role_arn=None):
    ctrl = boto3.client("bedrock", region_name=REGION)   # 'bedrock', not 'bedrock-runtime'
    cfg = {"cloudWatchConfig": {"logGroupName": log_group},
           "textDataDeliveryEnabled": True,
           "imageDataDeliveryEnabled": False,
           "embeddingDataDeliveryEnabled": False}
    if role_arn:
        cfg["cloudWatchConfig"]["roleArn"] = role_arn
    try:
        ctrl.put_model_invocation_logging_configuration(loggingConfig=cfg)
        print(f"Enabled. All Bedrock invocations now log to {log_group}.")
    except ClientError as e:
        print("Could not enable logging:", e.response["Error"]["Code"],
              "- ensure a role grants Bedrock logs:PutLogEvents on the group.")

# enable_bedrock_logging(role_arn="arn:aws:iam::123456789012:role/BedrockLoggingRole")
print("Call enable_bedrock_logging(...) once with a valid role ARN for consolidated logs.")

In [ ]:
# B. Push token/cost/latency to CloudWatch as custom metrics.
def emit_metrics(record, namespace="TravelMind/Bedrock"):
    cw = boto3.client("cloudwatch", region_name=REGION)
    dims = [{"Name": "Model", "Value": record["model"]}]
    md = [
        {"MetricName": "InputTokens",  "Dimensions": dims, "Value": record["input_tokens"], "Unit": "Count"},
        {"MetricName": "OutputTokens", "Dimensions": dims, "Value": record["output_tokens"], "Unit": "Count"},
        {"MetricName": "LatencyMs",    "Dimensions": dims, "Value": record["latency_ms"], "Unit": "Milliseconds"},
        {"MetricName": "CostUsd",      "Dimensions": dims, "Value": record["cost_usd"], "Unit": "None"},
    ]
    try:
        cw.put_metric_data(Namespace=namespace, MetricData=md)
        print(f"Pushed {len(md)} metrics to CloudWatch namespace {namespace}.")
    except ClientError as e:
        print("put_metric_data failed:", e.response["Error"]["Code"])

emit_metrics(rec)   # rec from the logging cell

In [ ]:
# C. EMF -- one JSON line that CloudWatch turns into metrics automatically.
def emf_log(record, namespace="TravelMind/Bedrock"):
    emf = {
        "_aws": {
            "Timestamp": int(time.time() * 1000),
            "CloudWatchMetrics": [{
                "Namespace": namespace,
                "Dimensions": [["Model"]],
                "Metrics": [{"Name": "InputTokens", "Unit": "Count"},
                            {"Name": "OutputTokens", "Unit": "Count"},
                            {"Name": "LatencyMs", "Unit": "Milliseconds"},
                            {"Name": "CostUsd", "Unit": "None"}]}]},
        "Model": record["model"],
        "InputTokens": record["input_tokens"], "OutputTokens": record["output_tokens"],
        "LatencyMs": record["latency_ms"], "CostUsd": record["cost_usd"],
        "request_id": record["request_id"],
    }
    print(json.dumps(emf))   # in Lambda/ECS this single print becomes logs + metrics

emf_log(rec)

### Consolidating it all — CloudWatch Logs Insights

Once your JSON/EMF logs land in one log group, aggregate across every call without touching the data:

```
fields @timestamp, model, input_tokens, output_tokens, cost_usd, latency_ms
| stats sum(input_tokens) as in_tok,
        sum(output_tokens) as out_tok,
        sum(cost_usd) as spend,
        avg(latency_ms) as avg_ms,
        pct(latency_ms, 99) as p99_ms
  by model
```

One query gives spend-by-model, total tokens, and p99 latency — the core observability view. Add CloudWatch **alarms** on `CostUsd` or `LatencyMs` to get paged before a runaway loop or latency regression bites.

## 9. Putting it together — a production-shaped client

One small class: config + retries + timeout + exception classification + structured logging + token/cost accounting + a bounded tool loop. The shape of what you would actually ship.

In [ ]:
class BedrockClient:
    def __init__(self, model_id=MODEL_ID, system=None):
        self.model_id = model_id
        self.system = [{"text": system}] if system else None
        self.total_cost = 0.0

    def _meter(self, resp):
        u = resp["usage"]
        p_in, p_out = PRICE.get(self.model_id, (0, 0))
        cost = u["inputTokens"] / 1e6 * p_in + u["outputTokens"] / 1e6 * p_out
        self.total_cost += cost
        logger.info(json.dumps({
            "ts": dt.datetime.utcnow().isoformat() + "Z", "model": self.model_id,
            "input_tokens": u["inputTokens"], "output_tokens": u["outputTokens"],
            "stop_reason": resp["stopReason"], "latency_ms": resp["metrics"]["latencyMs"],
            "cost_usd": round(cost, 6), "request_id": resp["ResponseMetadata"]["RequestId"]}))

    def call(self, messages, tool_config=None, max_tokens=1024, temperature=0.0):
        kw = dict(modelId=self.model_id, messages=messages,
                  inferenceConfig={"maxTokens": max_tokens, "temperature": temperature})
        if self.system:
            kw["system"] = self.system
        if tool_config:
            kw["toolConfig"] = tool_config
        resp = safe_converse(**kw)
        self._meter(resp)
        return resp

    def agent(self, user_text, tool_config, fns, max_turns=6):
        messages = [{"role": "user", "content": [{"text": user_text}]}]
        for _ in range(max_turns):
            resp = self.call(messages, tool_config=tool_config)
            messages.append(resp["output"]["message"])
            if resp["stopReason"] != "tool_use":
                return "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
            results = []
            for b in resp["output"]["message"]["content"]:
                if "toolUse" in b:
                    tu = b["toolUse"]
                    out = fns[tu["name"]](**tu["input"])
                    bad = isinstance(out, dict) and "error" in out
                    results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                                    "content": [{"json": out}],
                                    "status": "error" if bad else "success"}})
            messages.append({"role": "user", "content": results})
        return "stopped: max_turns"

client = BedrockClient(system="You are a concise airline assistant.")
print(client.agent("Find flights BLR to SIN on 2026-08-14.",
                   {"tools": [SEARCH_TOOL], "toolChoice": {"auto": {}}},
                   {"search_flights": search_flights}))
print()
print(f"session cost so far: ${client.total_cost:.6f}")

## Other features you will reach for

- **Guardrails:** pass `guardrailConfig={"guardrailIdentifier": ..., "guardrailVersion": ...}` to screen inputs/outputs (PII, topics, profanity). Create the guardrail once in the console/API first.
- **Provider-specific knobs:** things like Claude extended thinking or Nova thinking levels go in `additionalModelRequestFields` — check the model's docs for the exact field name, since it varies by provider.
- **Document chat:** the `document` block (pdf/csv/xlsx/...) lets you ask questions over a file in the same call as text.
- **Token counting before sending:** there is no separate count API for Converse; use the `usage` you get back to calibrate estimates, and cap with `maxTokens`.

## Cheat sheet

- Stateless: resend the full `messages` list every turn.
- `stopReason == "tool_use"` -> run tools -> return `toolResult` -> call again, until `end_turn`.
- Always loop over `content` blocks; the model may call several tools at once.
- Force structured output with `toolChoice: {"tool": {"name": ...}}` (Claude).
- Retry transient errors with backoff; raise permanent ones (`ValidationException`, `AccessDeniedException`).
- Log JSON per call (tokens, latency, cost, request id). Enable Bedrock invocation logging for zero-code capture.
- Push `put_metric_data` or EMF for dashboards/alarms; aggregate with Logs Insights.
- IAM action is `bedrock:InvokeModel`. Use the `us.` inference-profile IDs.